In [1]:
from pyspark.sql import (
    functions as f, 
    SparkSession, 
    types as t
)

In [7]:
spark = SparkSession.builder.appName('logical_plan_and_physical_plan').getOrCreate()
file_path = "file:///home/jovyan/work/sample/lorem_ipsum.txt"

df = spark.read.text(file_path)
df.show()
df.explain()

+--------------------+
|               value|
+--------------------+
|Lorem ipsum dolor...|
|                    |
|Orci eu lobortis ...|
|                    |
|Vulputate enim nu...|
|                    |
|Sit amet nulla fa...|
|                    |
|Nibh cras pulvina...|
|                    |
|Arcu felis bibend...|
|                    |
|Vestibulum sed ar...|
|                    |
|Sit amet tellus c...|
|                    |
|Augue mauris augu...|
|                    |
|Pellentesque mass...|
|                    |
+--------------------+
only showing top 20 rows

== Physical Plan ==
FileScan text [value#4] Batched: false, DataFilters: [], Format: Text, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/sample/lorem_ipsum.txt], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<value:string>




In [17]:
words = df.select(
    f.explode(f.split(df.value,' ')).alias('word')
)

words_counts = words.groupBy('word').count()
words_counts.explain(extended = True)

== Parsed Logical Plan ==
'Aggregate ['word], ['word, count(1) AS count#73L]
+- Project [word#69]
   +- Generate explode(split(value#4,  , -1)), false, [word#69]
      +- Relation [value#4] text

== Analyzed Logical Plan ==
word: string, count: bigint
Aggregate [word#69], [word#69, count(1) AS count#73L]
+- Project [word#69]
   +- Generate explode(split(value#4,  , -1)), false, [word#69]
      +- Relation [value#4] text

== Optimized Logical Plan ==
Aggregate [word#69], [word#69, count(1) AS count#73L]
+- Generate explode(split(value#4,  , -1)), [0], false, [word#69]
   +- Relation [value#4] text

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[word#69], functions=[count(1)], output=[word#69, count#73L])
   +- Exchange hashpartitioning(word#69, 200), ENSURE_REQUIREMENTS, [plan_id=137]
      +- HashAggregate(keys=[word#69], functions=[partial_count(1)], output=[word#69, count#77L])
         +- Generate explode(split(value#4,  , -1)), false, [word#69]
     

In [18]:
words_counts.explain(mode = 'formatted')

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Generate (2)
            +- Scan text  (1)


(1) Scan text 
Output [1]: [value#4]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/sample/lorem_ipsum.txt]
ReadSchema: struct<value:string>

(2) Generate
Input [1]: [value#4]
Arguments: explode(split(value#4,  , -1)), false, [word#69]

(3) HashAggregate
Input [1]: [word#69]
Keys [1]: [word#69]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#76L]
Results [2]: [word#69, count#77L]

(4) Exchange
Input [2]: [word#69, count#77L]
Arguments: hashpartitioning(word#69, 200), ENSURE_REQUIREMENTS, [plan_id=137]

(5) HashAggregate
Input [2]: [word#69, count#77L]
Keys [1]: [word#69]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#72L]
Results [2]: [word#69, count(1)#72L AS count#73L]

(6) AdaptiveSparkPlan
Output [2]: [word#69, count#73L]
Arguments: isFinalPlan=false


